# Agilent VSpin

The VSpin is a compact centrifuge from Agilent. It supports:

- [Centrifuging](../../capabilities/centrifuging) (1--1000 x g, configurable acceleration/deceleration)
- Bucket calibration and positioning
- Door and bucket lock control

It can optionally be paired with an **Access2** loader for automated plate loading/unloading.

## Setup

In [ ]:
from pylabrobot.agilent.vspin import VSpin

vspin = VSpin(name="vspin", device_id="YOUR_FTDI_ID_HERE")
await vspin.setup()

Find your FTDI device ID with:

```bash
python -m pylibftdi.examples.list_devices
```

See [Installation](#installation) below for driver setup instructions.

## Centrifuging

The VSpin exposes a {class}`~pylabrobot.capabilities.centrifuging.centrifuging.Centrifuge` on `vspin.centrifuging`. For the full API, see [Centrifuging](../../capabilities/centrifuging).

### Bucket positioning

Before using the go-to-bucket commands, you must calibrate the bucket positions (see [Calibrating bucket 1 position](#calibrating-bucket-1-position)).

In [ ]:
await vspin.centrifuging.go_to_bucket1()
await vspin.centrifuging.open_door()

In [ ]:
await vspin.centrifuging.go_to_bucket2()

### Spinning

Use {class}`~pylabrobot.agilent.vspin.VSpinCentrifugeBackend.SpinParams` to configure acceleration and deceleration (0--1 scale).

In [ ]:
from pylabrobot.agilent.vspin import VSpinCentrifugeBackend

await vspin.centrifuging.spin(
    g=800,
    duration=60,
    backend_params=VSpinCentrifugeBackend.SpinParams(
        acceleration=1.0,
        deceleration=1.0,
    ),
)

### Door and bucket control

In [ ]:
await vspin.centrifuging.open_door()
await vspin.centrifuging.close_door()
await vspin.centrifuging.lock_door()
await vspin.centrifuging.unlock_door()

In [ ]:
await vspin.centrifuging.lock_bucket()
await vspin.centrifuging.unlock_bucket()

### Status queries

The driver exposes low-level status queries:

In [ ]:
await vspin.driver.request_door_locked()
await vspin.driver.request_door_open()
await vspin.driver.request_bucket_locked()

## Calibrating bucket 1 position

You need to calibrate the bucket 1 position once per VSpin. The calibration is saved to disk at `~/.pylabrobot/vspin_bucket_calibrations.json` (keyed by USB serial number).

There are two ways to position bucket 1:

### Using code

Use `go_to_position` to rotate the buckets. A full rotation is 8000 ticks (4.5 degrees per 100 ticks).

In [ ]:
await vspin.centrifuging.backend.go_to_position(100)
await vspin.centrifuging.backend.set_bucket_1_position_to_current()

### Manual rotation

You can open the door, unlock the bucket, and manually rotate the buckets to align bucket 1 with the door.

```{warning}
The VSpin has a safety mechanism that closes the door when it detects movement. This triggers when you rotate the buckets manually too fast. Be careful -- it will save time compared to using code, but the door can close on your fingers.
```

In [ ]:
await vspin.centrifuging.open_door()
await vspin.centrifuging.unlock_bucket()
# Manually rotate buckets to align bucket 1 with door
await vspin.centrifuging.backend.set_bucket_1_position_to_current()

## Access2 loader

The VSpin can optionally be paired with an Access2 loader for automated plate loading/unloading. The loader is optional -- you can also use a robotic arm (e.g. iSWAP) to move plates directly into the centrifuge.

When using the loader, specify the FTDI device IDs for both the VSpin and the loader, since both use FTDI and are otherwise indistinguishable.

In [ ]:
import asyncio

from pylabrobot.agilent.vspin import VSpin, Access2

vspin = VSpin(name="vspin", device_id="YOUR_VSPIN_FTDI_ID_HERE")
loader = Access2(name="loader", device_id="YOUR_LOADER_FTDI_ID_HERE", vspin=vspin)

await asyncio.gather(vspin.setup(), loader.setup())

In [ ]:
# Go to a bucket and open the door before loading
await vspin.centrifuging.go_to_bucket1()
await vspin.centrifuging.open_door()

# Assign a plate to the loader (can also be done implicitly, e.g. lh.move_plate(plate, loader))
from pylabrobot.resources import Cor_96_wellplate_360ul_Fb
plate = Cor_96_wellplate_360ul_Fb(name="plate")
loader.assign_child_resource(plate)

# Load and unload
await loader.load()
await loader.unload()

## Teardown

In [ ]:
await vspin.stop()

## Installation

The VSpin connects via FTDI USB. You need the `libftdi` library installed on your system.

### macOS

```bash
brew install libftdi
```

### Linux (Debian/Ubuntu)

```bash
sudo apt-get install libftdi-dev
```

### Windows

1. Locate your Python `Scripts` folder (run `python -c "import sys; print(sys.executable)"` to find your Python directory).
2. Download the [FTDI Development Kit](https://sourceforge.net/projects/picusb/files/libftdi1-1.5_devkit_x86_x64_19July2020.zip/download).
3. Copy `libftdi1.dll` and `libusb-1.0.dll` from the `bin64` folder into your Python `Scripts` folder.
4. Use [Zadig](https://zadig.akeo.ie/) to replace the VSpin's default driver with `libusbk`. To identify the VSpin device, disconnect its RS232 cable while monitoring the Zadig device list -- the device that disappears is the VSpin.

```{note}
To revert to the original driver (e.g. for the Agilent Centrifuge Config Tool), uninstall the `libusbk` driver in Device Manager. The default driver will reinstall automatically.
```

### Finding the FTDI ID

```bash
python -m pylibftdi.examples.list_devices
```

This outputs something like `FTDI:USB Serial Converter:FTE0RJ5T`. The last field is the device ID.